# Kev-4B vs Kev-9B: 10-case checkpoint-verified pilot (Colab A100)

The previous BugsInPy 100-case comparison had **100/100 identical probability vectors** but no checkpoint attestation. This notebook reuses **10 historical test contexts from the saved 100-case run** and verifies the actual local Kev server's `GET /v1/models` `run` identity for each model. It runs 9B then 4B sequentially on **one A100**, never a paid Claude API.

**Use a fresh Colab runtime with A100 40GB+**, choose **Runtime → Run all**. The original 100 cases are read-only. This pilot creates a new Google Drive folder and saves the selected IDs, checkpoint attestations, all choice-probability vectors, complete comparison and server startup logs. Even exact equality after both model identities are verified would require further investigation; it is not an accuracy result.

In [ ]:
import json, os, subprocess, sys, time, uuid
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
SOURCE_RUN_ID = "kev_a100_20261026_083544_UTC_c09a65"  # previous 100-case audit source
RESULTS = Path("/content/drive/MyDrive/adaptive-llm-router/results")
SOURCE_RUN = RESULTS / SOURCE_RUN_ID
SOURCE_CASES = SOURCE_RUN / "outputs/bugsinpy_cases.jsonl"
if not SOURCE_CASES.is_file():
    raise FileNotFoundError(f"Previous 100-case input is missing: {SOURCE_CASES}")
PILOT_ID = "kev_identity_pilot_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC") + "_" + uuid.uuid4().hex[:6]
PILOT_ROOT = RESULTS / PILOT_ID
OUTPUTS = PILOT_ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=False)
(PILOT_ROOT / "logs").mkdir()
print("Previous source:", SOURCE_CASES)
print("New pilot folder:", PILOT_ROOT)
print("Original source will not be modified.")

In [ ]:
# Minimal local installation for Kev server and existing GitHub router utilities.
import hashlib, random, shutil, importlib
gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                     text=True, capture_output=True, check=True)
memory_mib = int(gpu.stdout.strip().splitlines()[0])
if memory_mib < 30000:
    raise RuntimeError("Use A100 40GB or 80GB GPU; current VRAM is insufficient for Kev-9B.")
print("Available GPU VRAM capacity:", memory_mib, "MiB")

ROOT = Path("/content/adaptive-llm-router-identity-pilot")
UPSTREAM = Path("/content/kev-identity-pilot")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--quiet",
                    "https://github.com/namyikim/adaptive-llm-router.git", str(ROOT)],
                   check=True, timeout=300)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
PROJECT_SHA = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
from adaptive_llm_router.bugsinpy_compare import (read_cases, read_rows, collect_decisions,
                                                    compare_decisions, probe_model_identity)
from adaptive_llm_router.routing import TIERS

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    raise RuntimeError("uv not installed")
if not (UPSTREAM / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--quiet",
                    "https://github.com/jaredpalmer/kev.git", str(UPSTREAM)],
                   check=True, timeout=300)
KEV_SOURCE_SHA = subprocess.check_output(["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([uv, "sync", "--extra", "serve"], cwd=UPSTREAM, check=True, timeout=1200)
print("Adaptive router commit:", PROJECT_SHA)
print("Kev upstream commit:", KEV_SOURCE_SHA)

In [ ]:
# Pick exactly 10 DISTINCT cases from multiple projects, without reading any
# historical model predictions to select them.
from collections import defaultdict
old_cases = read_cases(SOURCE_CASES)
grouped = defaultdict(list)
for case in old_cases:
    grouped[case["id"].rsplit("-", 1)[0]].append(case)
for project, records in grouped.items():
    random.Random(int.from_bytes(
        hashlib.sha256(("20261026/" + project).encode()).digest()[:8], "big")
    ).shuffle(records)
selected = []
while len(selected) < 10 and any(grouped.values()):
    for project in sorted(grouped):
        if grouped[project] and len(selected) < 10:
            selected.append(grouped[project].pop())
if len(selected) != 10:
    raise RuntimeError(f"Only {len(selected)} usable cases, expected 10")

pilot_cases = OUTPUTS / "pilot_cases.jsonl"
with pilot_cases.open("w", encoding="utf-8") as f:
    for case in selected:
        f.write(json.dumps(case, ensure_ascii=False) + "\n")
    f.flush()
    os.fsync(f.fileno())

pilot_manifest = {
    "source_run_id": SOURCE_RUN_ID, "source_100_cases_sha256": hashlib.sha256(SOURCE_CASES.read_bytes()).hexdigest(),
    "selection": "deterministic project-balanced 10 cases; no prior model probabilities used",
    "cases": [{"id": x["id"], "input_sha256": x["input_sha256"]} for x in selected],
    "router_commit": PROJECT_SHA, "kev_source_commit": KEV_SOURCE_SHA,
    "expected_checkpoints": ["jaredpalmer/kev-9b@v1.0", "jaredpalmer/kev-4b@v1.0"],
}
(OUTPUTS / "pilot_manifest.json").write_text(
    json.dumps(pilot_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Selected IDs:", [case["id"] for case in selected])
print("Project count:", len({case["id"].rsplit("-", 1)[0] for case in selected}))

In [ ]:
# Start ONE Kev server at a time and independently verify /v1/models.run.
import urllib.request, urllib.error
from adaptive_llm_router.bugsinpy_compare import probe_model_identity, KevModelMismatch
PORT = 8009
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_ROOT = PILOT_ROOT / "logs"

def start_verified_checkpoint(checkpoint_run, label, timeout_seconds=1200):
    path = LOG_ROOT / (label.replace("@", "_") + ".log")
    with path.open("a", encoding="utf-8") as logfile:
        process = subprocess.Popen(
            [uv, "run", "--extra", "serve", "python", "-m", "kev.serve",
             "--run", checkpoint_run, "--device", "cuda",
             "--host", "127.0.0.1", "--port", str(PORT)],
            cwd=UPSTREAM, stdout=logfile, stderr=subprocess.STDOUT
        )
    started = time.monotonic()
    proof = None
    while time.monotonic() - started < timeout_seconds:
        if process.poll() is not None:
            raise RuntimeError(f"{checkpoint_run} process exited. Check {path}")
        try:
            proof = probe_model_identity(BASE_URL, label)
            break
        except (KevModelMismatch, OSError, TimeoutError):
            time.sleep(10)
    if not proof:
        process.terminate()
        raise TimeoutError(f"Kev model {checkpoint_run} did not become ready: {path}")
    proof.update({"pid": process.pid, "log_file": path.name, "checkpoint_run_requested": checkpoint_run})
    print("Verified GPU model:", proof["reported_run"], "PID:", process.pid)
    return process, proof

def stop_checkpoint(process):
    if process and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=45)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=15)

def save_json(path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

In [ ]:
# Pilot: 9B -> shut down -> 4B. Each model verifies the LIVE checkpoint before
# collecting choices. A wrong checkpoint causes a FAILED experiment, never a
# mislabeled successful report. Results remain in Drive after any interruption.
decisions = OUTPUTS / "pilot_decisions.jsonl"
attestations = OUTPUTS / "model_attestations.json"
status_file = PILOT_ROOT / "RUN_STATUS.json"
active = None
proofs = {}
run_status = {"pilot_id": PILOT_ID, "phase": "starting", "completed": False}
save_json(status_file, run_status)
try:
    for label in ("kev-9b@v1.0", "kev-4b@v1.0"):
        expected_run = "jaredpalmer/" + label
        run_status["phase"] = "loading_" + label
        save_json(status_file, run_status)
        active, proof = start_verified_checkpoint(expected_run, label)
        proofs[label] = proof
        save_json(attestations, proofs)
        run_status["phase"] = "routing_" + label
        save_json(status_file, run_status)
        result = collect_decisions(
            pilot_cases, decisions, label,
            base_url=BASE_URL,
        )
        print("Checkpoint:", label, "proposed decisions:", result["new_successes"])
        run_status["completed_" + label] = result["new_successes"]
        save_json(status_file, run_status)
        stop_checkpoint(active)
        active = None

    if proofs["kev-4b@v1.0"]["reported_run"] == proofs["kev-9b@v1.0"]["reported_run"]:
        raise RuntimeError("Both live model identity reports unexpectedly match")
    analysis = compare_decisions(read_cases(pilot_cases), read_rows(decisions))
    save_json(OUTPUTS / "pilot_comparison.json", analysis)
    if analysis["n_paired"] != len(selected):
        raise RuntimeError(f"Incomplete verified comparison: {analysis['n_paired']} / {len(selected)}")
    n_exact = analysis["exact_probability_vector_match_count"]
    conclusion = ("all_10_identical_requires_investigation" if n_exact == 10
                  else "different_probabilities_with_checkpoint_attestation")
    summary = {
        "status": conclusion,
        "n_paired": analysis["n_paired"],
        "n_exact_probability_vector_matches": n_exact,
        "n_raw_disagreement": analysis["n_raw_disagreement"],
        "n_final_disagreement": analysis["n_final_disagreement"],
        "n_model_identity_verified": analysis["model_identity_verified_counts"],
        "checkpoint_attestations": proofs,
        "notes": "Identical probability vectors or differing ones are not bug-fix correctness scores.",
    }
    save_json(OUTPUTS / "pilot_summary.json", summary)
    run_status["phase"] = "finished"
    run_status["completed"] = True
    save_json(status_file, run_status)
    save_json(PILOT_ROOT / "SAVE_COMPLETE.json", {
        "pilot_id": PILOT_ID, "result": conclusion, "n_paired": analysis["n_paired"]})
    print("\n*** PILOT RESULT ***")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
except Exception as exc:
    run_status["phase"] = "failed"
    run_status["error_type"] = type(exc).__name__
    run_status["completed"] = False
    save_json(status_file, run_status)
    raise
finally:
    stop_checkpoint(active)


In [ ]:
## Review and share

The result is in **MyDrive/adaptive-llm-router/results/`kev_identity_pilot_...`/outputs/pilot_summary.json**.

- Both models must report **different, expected** checkpoint IDs in `model_attestations.json`.
- `n_exact_probability_vector_matches` counts exact matches of all three L1/L2/L3 choices, not rounded averages.
- If 10/10 probability vectors are equal even with verified checkpoint IDs, investigate model outputs and checkpoints further. This still does **not** measure LLM bug resolution quality.
- Share `pilot_summary.json` and `pilot_comparison.json` for review. These contain no raw test code; the public 10-case inputs are saved separately.

When this pilot establishes trustworthy model identities and differentiated outputs, proceed to the larger 100-case experiment.